# Build per-repo FAISS indexes from guideline chunks

Loads repo-specific chunk JSONs, creates embeddings, builds FAISS indexes, saves index + metadata, and demos retrieval.
Embedding model: BAAI/bge-large-en-v1.5 (replace if you use another client)

In [1]:
# If needed, uncomment and run once in your environment
# !pip install -q sentence-transformers faiss-cpu numpy

In [2]:
from pathlib import Path
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

ROOT = Path('..').resolve()
CHUNKS_DIR = ROOT / 'data' / 'processed' / 'guideline_chunks'
OUT_DIR = ROOT / 'data' / 'processed' / 'faiss_db'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'BAAI/bge-large-en-v1.5'
model = SentenceTransformer(MODEL_NAME)


c:\Users\budhi\Documents\IITM\DSAI Lab\Group-1-DS-and-AI-Lab-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1973.62it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
def build_faiss_for_repo(chunk_file, out_dir=OUT_DIR, normalize=True):
    with open(chunk_file, 'r', encoding='utf-8') as f:
        chunks = json.load(f)

    texts = [c['text'] for c in chunks]
    metadata = [
        {
            'chunk_id': c.get('chunk_id'),
            'repo': c.get('repo'),
            'source_file': c.get('source_file'),
            'source_id': c.get('source_id'),
            'source_url': c.get('source_url'),
            'source_title': c.get('source_title'),
            'section_hint': c.get('section_hint'),
            'word_count': c.get('word_count'),
        }
        for c in chunks
    ]

    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=normalize,
    ).astype('float32')

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim) if normalize else faiss.IndexFlatL2(dim)
    index.add(embeddings)

    repo_name = Path(chunk_file).stem.replace('_chunks', '')
    idx_path = out_dir / f'{repo_name}_faiss_index.bin'
    meta_path = out_dir / f'{repo_name}_faiss_metadata.json'

    faiss.write_index(index, str(idx_path))
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    return {
        'repo': repo_name,
        'n_vectors': index.ntotal,
        'index_path': str(idx_path),
        'meta_path': str(meta_path),
    }


In [4]:
results = []
for p in sorted(CHUNKS_DIR.glob('*_chunks.json')):
    print('Building:', p.name)
    r = build_faiss_for_repo(p)
    results.append(r)

print('Done. Built indexes for:', [r['repo'] for r in results])


Building: django_chunks.json


Batches: 100%|██████████| 1/1 [01:16<00:00, 76.45s/it]


Building: fastapi_chunks.json


Batches: 100%|██████████| 1/1 [00:24<00:00, 24.27s/it]


Building: pandas_chunks.json


Batches: 100%|██████████| 1/1 [01:58<00:00, 118.63s/it]


Building: scikit-learn_chunks.json


Batches: 100%|██████████| 1/1 [00:33<00:00, 33.17s/it]

Done. Built indexes for: ['django', 'fastapi', 'pandas', 'scikit-learn']


In [5]:
def load_index_and_meta(index_path, meta_path):
    idx = faiss.read_index(str(index_path))
    with open(meta_path, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    return idx, meta

def retrieve_from_index(idx, meta, query, top_k=5):
    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    scores, indices = idx.search(q_emb, top_k)
    out = []
    for score, i in zip(scores[0], indices[0]):
        item = meta[i]
        out.append({'score': float(score), 'meta': item})
    return out

# Demo: pick first built index (if any)
if results:
    example = results[0]
    idx, meta = load_index_and_meta(example['index_path'], example['meta_path'])
    hits = retrieve_from_index(idx, meta, "How should I handle missing values in pandas?", top_k=5)
    for h in hits:
        print(h['score'], h['meta']['chunk_id'], h['meta']['source_file'])
        print('---')
else:
    print('No indexes built; run the previous cell to build indexes.')


0.5480432510375977 django_001__2 data\raw\guidelines_raw\django_guidelines_raw.md
---
0.539228081703186 django_001__3 data\raw\guidelines_raw\django_guidelines_raw.md
---
0.538450300693512 django_001__1 data\raw\guidelines_raw\django_guidelines_raw.md
---
0.5254778861999512 django_005__1 data\raw\guidelines_raw\django_guidelines_raw.md
---
0.5247170925140381 django_002__2 data\raw\guidelines_raw\django_guidelines_raw.md
---
